# Анализ заказов интернет-магазина

Учебный проект на искусственных данных. Здесь я разбираю загрузку CSV, поиск повторов, простые расчёты в Python и группировку в SQL.

## 1. Открываю таблицу
Загружаю файл `orders_raw.csv` и смотрю первые строки.

In [ ]:
import io
import pandas as pd
from google.colab import files

uploaded = files.upload()
raw = pd.read_csv(io.BytesIO(next(iter(uploaded.values()))))
print('Всего строк:', len(raw))
raw.head()

## 2. Проверяю повторы и пропуски
Один заказ должен учитываться один раз. Поэтому проверяю `order_id` и удаляю повторяющиеся строки.

In [ ]:
print('Повторы заказов:', raw['order_id'].duplicated().sum())
print('Пропуски по столбцам:')
print(raw.isna().sum())

df = raw.drop_duplicates(subset='order_id').copy()
print('Осталось заказов:', len(df))

## 3. Считаю основные показатели
Разделяю заказы по статусам и отдельно считаю стоимость только доставленных.

In [ ]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['month'] = df['order_date'].dt.strftime('%Y-%m')
df['order_count'] = 1
df['is_delivered'] = (df['status'] == 'delivered').astype(int)
df['is_finished'] = df['status'].isin(['delivered', 'cancelled']).astype(int)
df['delivered_amount'] = df['amount_rub'].where(df['status'] == 'delivered', 0)

print(df['status'].value_counts())
print('Стоимость доставленных заказов:', df['delivered_amount'].sum())

## 4. Смотрю категории с помощью SQL
`GROUP BY` объединяет заказы одной категории, `COUNT` считает их количество, а `SUM` складывает стоимость доставленных.

In [ ]:
import sqlite3
conn = sqlite3.connect(':memory:')
df.to_sql('orders', conn, index=False, if_exists='replace')

query = '''
SELECT category,
       COUNT(*) AS orders_count,
       SUM(delivered_amount) AS delivered_sum
FROM orders
GROUP BY category
ORDER BY delivered_sum DESC;
'''
pd.read_sql_query(query, conn)

## 5. Смотрю заказы по месяцам
Отдельно сравниваю количество всех и доставленных заказов по месяцам.

In [ ]:
query = '''
SELECT month,
       COUNT(*) AS all_orders,
       SUM(is_delivered) AS delivered_orders
FROM orders
GROUP BY month
ORDER BY month;
'''
pd.read_sql_query(query, conn)

## 6. Сохраняю таблицу
Готовый CSV можно загрузить в DataLens для построения графиков.

In [ ]:
df['order_date'] = df['order_date'].dt.strftime('%Y-%m-%d')
df.to_csv('orders_bi.csv', index=False, encoding='utf-8-sig')
files.download('orders_bi.csv')

## Краткий результат
После удаления 12 повторов осталось 500 заказов. Из них 367 доставлены, 74 отменены и 59 находятся в обработке. Стоимость доставленных заказов составила 2 096 862 ₽. Данные учебные, поэтому выводы относятся только к этому примеру.